# Inline narrative numbers

Companion to `REPORT_final_FINAL.ipynb`. This notebook prints aggregate values
and small tables for quoting in the thesis text — no figures.

Shared analysis constants match the main report notebook so numbers are
directly comparable to the curated figures. Heavy results are loaded from
`local/_cache/` when available (same files as `REPORT_final_FINAL.ipynb`).


In [1]:
%load_ext autoreload
%autoreload 2

## Setup

In [2]:
import sys
from pathlib import Path

from IPython.display import display

for _path in (Path.cwd(), *Path.cwd().parents):
    if (_path / "pyproject.toml").is_file():
        PROJECT_ROOT = _path
        NOTEBOOK_DIR = Path.cwd()
        for p in (str(_path / "src"), str(_path / "local"), str(_path)):
            if p not in sys.path:
                sys.path.insert(0, p)
        break
else:
    raise RuntimeError("Could not find project root (pyproject.toml)")

from single_room_multi_opt_compute import OPTIMIZER_ORDER, USE_GAUSSIANS
from inline_complements import (
    InlineAnalysisConfig,
    PF_LENGTH_ROUND,
    SINGLE_ROOM_ALIVE_PROPORTION_FIT_COLUMNS,
    SINGLE_ROOM_PF_LENGTH_COLUMNS,
    TWO_ROOMS_MEAN_R2_FIT_COLUMNS,
    compute_single_room_alive_proportion_fit,
    compute_single_room_pf_length_summary,
    compute_two_rooms_train_same_mean_r2_fit,
    format_linear_fit_latex_table,
    format_linear_fit_table,
    load_single_room_results,
    load_two_rooms_results,
)

from report_final.notebook_cache import report_final_cache_dir

RESULTS_PARENT_OVERRIDE = None
ANALYSIS_CONFIG = InlineAnalysisConfig()
CACHE_DIR = report_final_cache_dir(PROJECT_ROOT)
USE_CACHE = True
FORCE_RECOMPUTE = False

if not USE_GAUSSIANS:
    raise RuntimeError("This notebook always uses Gaussian RF fits.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Optimizers:   {OPTIMIZER_ORDER}")
print(f"Cache dir:    {CACHE_DIR} (USE_CACHE={USE_CACHE})")


Project root: /home/garvalov/thesis/plateau_optimizer/place_cells_episodic_rnn
Optimizers:   ['sgd', 'adagrad', 'adam', 'pure_shampoo', 'grafted_shampoo']
Cache dir:    /home/garvalov/thesis/plateau_optimizer/place_cells_episodic_rnn/local/_cache (USE_CACHE=True)


## Alive proportion linear fit — single room

Linear fit of proportion alive vs global capture index, matching
`plot_alive_proportion_by_optimizer` (figure `03_single_room_alive_proportion`),
starting after the first trajectory (`traj_id >= 1`).

- **slope** — change in proportion alive per capture
- **intercept** — fitted proportion alive at capture index 0
- **fit_r2** — coefficient of determination for the linear model

In [3]:
single_results = load_single_room_results(
    project_root=PROJECT_ROOT,
    results_parent_override=RESULTS_PARENT_OVERRIDE,
    analysis_config=ANALYSIS_CONFIG,
    cache_dir=CACHE_DIR,
    use_cache=USE_CACHE,
    force_recompute=FORCE_RECOMPUTE,
)


Loaded single-room results from cache: /home/garvalov/thesis/plateau_optimizer/place_cells_episodic_rnn/local/_cache/single_room.pkl.gz


In [4]:
single_room_alive_proportion_fit = compute_single_room_alive_proportion_fit(single_results)

display(
    format_linear_fit_table(
        single_room_alive_proportion_fit,
        columns=SINGLE_ROOM_ALIVE_PROPORTION_FIT_COLUMNS,
    )
)

,optimizer_label,n_points,slope,intercept,fit_r2
0,SGD,610,0.000011,0.3139,0.0216
1,AdaGrad,610,0.000157,0.2733,0.9039
2,Adam,610,-0.000038,0.2775,0.3460
3,Pure Shampoo,610,-0.000142,0.2744,0.8257
4,Grafted Shampoo,610,-0.000116,0.2860,0.8615


## Mean PF length — single room

Mean place-field length (seconds) per optimizer, matching the length panel in
`plot_pf_metric_bars_by_optimizer` (active segments only; mean ± 1.96 SEM).

In [ ]:
single_room_pf_length = compute_single_room_pf_length_summary(single_results)

display(
    format_linear_fit_table(
        single_room_pf_length,
        columns=SINGLE_ROOM_PF_LENGTH_COLUMNS,
        round_spec=PF_LENGTH_ROUND,
    )
)

print("\nMean PF length (s) — single room:")
for row in single_room_pf_length.itertuples(index=False):
    print(f"  {row.optimizer_label}: {row.mean_length_s:.2f} ± {row.ci95_length_s:.2f} (n={row.n_pfs})")

## Mean R² linear fit — two rooms (train_same only)

Same metric as `plot_dual_room_metric_by_optimizer(..., y_col="mean_r2")`, but
restricted to **train_same** captures (`visit_room_id == eval_room_id`), starting
after the first repetition (`rep_id >= 1`). One fit per optimizer, pooling all
train_same points across both eval rooms.

In [8]:
two_rooms_results = load_two_rooms_results(
    project_root=PROJECT_ROOT,
    results_parent_override=RESULTS_PARENT_OVERRIDE,
    analysis_config=ANALYSIS_CONFIG,
    cache_dir=CACHE_DIR,
    use_cache=USE_CACHE,
    force_recompute=FORCE_RECOMPUTE,
)


Loaded two-rooms results from cache: /home/garvalov/thesis/plateau_optimizer/place_cells_episodic_rnn/local/_cache/two_rooms.pkl.gz


In [9]:
two_rooms_mean_r2_fit = compute_two_rooms_train_same_mean_r2_fit(two_rooms_results)

display(
    format_linear_fit_table(
        two_rooms_mean_r2_fit,
        columns=TWO_ROOMS_MEAN_R2_FIT_COLUMNS,
    )
)

,optimizer_label,n_points,slope,intercept,fit_r2
0,SGD,960,0.000012,0.7747,0.4480
1,AdaGrad,960,0.000082,0.6854,0.9044
2,Adam,960,0.000061,0.6494,0.7753
3,Pure Shampoo,960,-0.000043,0.7037,0.4515
4,Grafted Shampoo,960,-0.000013,0.7353,0.1526


## LaTeX table — linear fit slopes

Combined booktabs table for pasting into the thesis (grouped by experiment category).

In [ ]:
linear_fit_latex = format_linear_fit_latex_table(
    single_room_alive_proportion_fit,
    two_rooms_mean_r2_fit,
    experiment_order=[
        "single_room_alive_proportion",
        "two_rooms_train_same",
    ],
)

print(linear_fit_latex)

\begin{table}[ht]
\centering
\small
\resizebox{0.55\linewidth}{!}{%
\begin{tabular}{llcc}
\toprule
Category & Optimizer & Slope & $R^2_{\mathrm{fit}}$ \\
\midrule
Single room & SGD & $1.1 \times 10^{-5}$ & 0.02165 \\
 & AdaGrad & $1.6 \times 10^{-4}$ & 0.9039 \\
 & Adam & $-3.8 \times 10^{-5}$ & 0.346 \\
 & Pure~Shampoo & $-1.4 \times 10^{-4}$ & 0.8257 \\
 & Grafted~Shampoo & $-1.2 \times 10^{-4}$ & 0.8615 \\
Two rooms (train\_same) & SGD & $1.2 \times 10^{-5}$ & 0.448 \\
 & AdaGrad & $8.2 \times 10^{-5}$ & 0.9044 \\
 & Adam & $6.1 \times 10^{-5}$ & 0.7753 \\
 & Pure~Shampoo & $-4.3 \times 10^{-5}$ & 0.4515 \\
 & Grafted~Shampoo & $-1.3 \times 10^{-5}$ & 0.1526 \\
\bottomrule
\end{tabular}%
}
\caption{Linear fits of timeline metrics vs.\ global capture index: slope (change per capture) and coefficient of determination $R^2_{\mathrm{fit}}$ of the ordinary least-squares model.}
\label{tab:linear_fit_timelines}
\end{table}
